# Chapter 11: Large Language Models & Transformers
## Notebook 01 — Transformer Architecture

This notebook builds the **Transformer** from first principles. We start from the limitations of RNNs that motivated attention, implement **scaled dot-product attention**, generalise to **multi-head attention**, add **sinusoidal positional encoding**, stack everything into a **transformer encoder block**, and contrast the encoder / decoder / encoder–decoder families that produce BERT-, GPT- and T5-style models.

### What you'll learn

| Topic | Section |
|-------|--------|
| RNN limitations and why attention helps | §2 |
| Scaled dot-product attention (NumPy) | §3 |
| Multi-head attention (NumPy) | §4 |
| Sinusoidal positional encoding | §5 |
| Transformer encoder block | §6 |
| Encoder vs decoder vs encoder–decoder | §7 |
| Tokenization (BPE / WordPiece) intuition | §8 |

**Estimated time:** 3 hours

---
*Generated by Berta AI*

---
## 1. Introduction & Setup

We will rely only on **NumPy** so the math is fully transparent. The optional ``transformers`` import in §8 is wrapped in ``try/except`` and falls back to a manual demo if Hugging Face is not installed.

In [ ]:
import sys
import os
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'scripts'))

import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline
plt.rcParams['figure.figsize'] = (8, 4)
np.random.seed(42)

print("Setup complete.")

---
## 2. From RNNs to Attention

Recurrent networks (Chapter 9, Chapter 10) process tokens **sequentially**:

- **Long-range dependencies** vanish: information from token *t-100* must survive 100 hidden-state updates.
- **No parallelism** in time: each step depends on the previous step.
- **Fixed bottleneck**: a single hidden vector summarises everything.

**Attention** lets every output token look directly at every input token. The Transformer (Vaswani et al., 2017) drops recurrence entirely — it is *only* attention plus feed-forward layers.

In [ ]:
# Tiny demo: an "RNN" that has to compress a long sequence into one vector.
seq_len, hidden = 50, 8
x = np.random.randn(seq_len, hidden)
h = np.zeros(hidden)
W = np.random.randn(hidden, hidden) * 0.1
U = np.random.randn(hidden, hidden) * 0.1
for t in range(seq_len):
    h = np.tanh(h @ W + x[t] @ U)
print("Final RNN state norm:", np.linalg.norm(h).round(3))
print("Information from x[0] is squashed through 50 non-linearities before reaching the output.")

---
## 3. Scaled Dot-Product Attention

Given queries ``Q``, keys ``K`` and values ``V``:

$$\text{Attention}(Q,K,V) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right)V$$

- ``QK^T`` measures how well each query matches each key.
- The ``1/√d_k`` factor stops dot products growing with dimensionality.
- ``softmax`` turns scores into probabilities — a **soft lookup**.

In [ ]:
from transformer_utils import scaled_dot_product_attention, softmax

# 4 query positions, 4 key/value positions, d_k = d_v = 8
Q = np.random.randn(4, 8)
K = np.random.randn(4, 8)
V = np.random.randn(4, 8)

out, weights = scaled_dot_product_attention(Q, K, V)
print("Output shape:", out.shape)         # (4, 8)
print("Attention weights shape:", weights.shape)  # (4, 4)
print("Each row sums to 1:", weights.sum(axis=1).round(4))

In [ ]:
# Visualise the attention matrix.
fig, ax = plt.subplots(figsize=(4, 3.5))
im = ax.imshow(weights, cmap='viridis')
ax.set_xlabel('Key position'); ax.set_ylabel('Query position')
ax.set_title('Self-attention weights')
fig.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

---
## 4. Multi-Head Attention

A single attention head has limited expressivity — it can only learn one relationship pattern. **Multi-head attention** runs ``h`` parallel attention computations on linear projections of the input, then concatenates and projects the result.

$$\text{MultiHead}(X) = \text{Concat}(\text{head}_1, \dots, \text{head}_h) W^O$$

Each head sees a different subspace; together they capture syntax, coreference, position, semantics and more.

In [ ]:
from transformer_utils import MultiHeadAttention

batch, seq_len, d_model, num_heads = 1, 6, 32, 4
x = np.random.randn(batch, seq_len, d_model).astype(np.float32)

mha = MultiHeadAttention(d_model=d_model, num_heads=num_heads, seed=0)
y = mha(x)
print("Input  shape:", x.shape)
print("Output shape:", y.shape)
print("Per-head attention weights:", mha.last_attn_weights.shape)
# (batch=1, num_heads=4, seq=6, seq=6)

In [ ]:
# Show that different heads attend to different positions.
fig, axes = plt.subplots(1, num_heads, figsize=(12, 3))
for h in range(num_heads):
    axes[h].imshow(mha.last_attn_weights[0, h], cmap='viridis')
    axes[h].set_title(f'Head {h}')
    axes[h].set_xticks([]); axes[h].set_yticks([])
plt.suptitle('Each head learns a different attention pattern')
plt.tight_layout()
plt.show()

---
## 5. Positional Encoding

Self-attention is **permutation-equivariant** — shuffle the tokens and the output shuffles the same way. Language obviously has order, so we *inject* positional information by adding a positional vector to each token embedding.

Sinusoidal positional encoding uses a different frequency per dimension:

$$PE_{pos,2i} = \sin(pos / 10000^{2i/d_{model}})$$
$$PE_{pos,2i+1} = \cos(pos / 10000^{2i/d_{model}})$$

This generalises to longer sequences than seen in training and gives the model relative-position information for free.

In [ ]:
from transformer_utils import positional_encoding

pe = positional_encoding(seq_len=64, d_model=32)
print("PE shape:", pe.shape)

fig, ax = plt.subplots(figsize=(8, 3.5))
im = ax.imshow(pe, aspect='auto', cmap='RdBu')
ax.set_xlabel('Embedding dim'); ax.set_ylabel('Position')
ax.set_title('Sinusoidal positional encoding')
fig.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

---
## 6. The Transformer Encoder Block

One encoder block is:

```
x = LayerNorm(x + MultiHeadAttention(x))
x = LayerNorm(x + FeedForward(x))
```

The **residual connections** preserve gradient flow; **layer norm** keeps activations stable; the position-wise **feed-forward** network gives each token a non-linear transformation.

In [ ]:
from transformer_utils import TransformerBlock

block = TransformerBlock(d_model=32, num_heads=4, ffn_hidden=64, seed=1)
x = np.random.randn(1, 10, 32).astype(np.float32) + positional_encoding(10, 32)
y = block(x)
print("Encoder block input shape :", x.shape)
print("Encoder block output shape:", y.shape)

# Stacking N blocks is the full encoder of e.g. BERT (N=12 for BERT-base).
y2 = block(y)  # call twice -> "2-layer encoder"
print("After 2 blocks            :", y2.shape)

---
## 7. Encoder, Decoder, Encoder–Decoder

| Family | Examples | Attention | Best for |
|--------|----------|-----------|----------|
| **Encoder-only** | BERT, RoBERTa, DistilBERT | Bidirectional | Classification, NER, embeddings |
| **Decoder-only** | GPT-2/3/4, Llama, Mistral | Causal (masked) | Text generation, chat |
| **Encoder–decoder** | T5, BART, mT5 | Bidirectional encoder + causal decoder with cross-attention | Translation, summarisation |

The **causal mask** is what makes a decoder autoregressive — at position *t* it can only attend to positions ``≤ t``.

In [ ]:
from transformer_utils import causal_mask
mask = causal_mask(6)
print("Causal mask (1 = allowed, 0 = blocked):")
print(mask)

# Apply it to attention scores
Q = np.random.randn(6, 8)
out, weights = scaled_dot_product_attention(Q, Q, Q, mask=mask)
print("\nMasked attention weights (upper triangle is zero):")
print(weights.round(2))

---
## 8. Tokenization: BPE & WordPiece

Modern LLMs do **not** tokenize on whitespace. They use **subword** algorithms:

- **BPE (Byte-Pair Encoding)** — used by GPT-2/3/4, RoBERTa. Start with bytes, iteratively merge the most frequent adjacent pair.
- **WordPiece** — used by BERT/DistilBERT. Same idea, slightly different merge criterion.
- **SentencePiece / Unigram** — used by T5, Llama. Treats the input as a raw byte stream.

Subwords give a small vocabulary (~30k–50k) with **no OOV** problem: any string can be encoded by falling back to characters.

In [ ]:
# Try the real tokenizer; fall back to a manual demo if transformers is not installed.
try:
    from transformers import AutoTokenizer
    tok = AutoTokenizer.from_pretrained('distilbert-base-uncased')
    text = "Transformers tokenize subwords like 'tokenization' -> ['token', '##ization']."
    ids = tok.encode(text)
    pieces = tok.tokenize(text)
    print("Pieces:", pieces)
    print("IDs   :", ids[:15], "...")
except Exception as e:
    print(f"transformers not installed ({e}); showing manual BPE intuition.")
    text = "tokenization"
    # Pretend our learned merges are: t+o, k+e, n+i, z+a, ti+on
    pieces = ['token', '##iz', '##ation']
    print(f"'{text}' -> {pieces}")
    print("With ## marking continuation pieces (WordPiece convention).")

---
## 9. Key Takeaways

- **Attention** replaces recurrence: every position attends to every other position in *parallel*.
- **Scaled dot-product** + **multi-head** + **positional encoding** + **residual + layer-norm + FFN** = one transformer block.
- The same block, masked or unmasked, gives encoder-only (BERT), decoder-only (GPT) or encoder–decoder (T5) models.
- Tokenization is **subword**: BPE / WordPiece / SentencePiece keep vocabularies small without OOV.

Next: **Notebook 02** — load real pretrained transformers, extract embeddings, and build classifiers on top of them.

---
*Generated by Berta AI*